# B.O.S.S. — Confronto unificato Object Detector (OR4.3 — P3)

Confronto di **quattro** object detector **a parità di risoluzione di input** a partire
da un frame RGB 640×480 (Intel RealSense D435i o cartella di test).

| Modello | File | Runtime |
|---|---|---|
| YOLO11n | `.pt` | `ultralytics` |
| EfficientDet Lite4 | `.tflite` | `tensorflow.lite` |
| NanoDet-Plus-m | `.onnx` | `onnxruntime` |
| PicoDet-S | `.onnx` | `onnxruntime` |

## Perché questo notebook
Il confronto precedente valutava ogni modello alla **propria risoluzione nativa**
(640 / 416 / 320 px): impossibile separare il contributo della **risoluzione** da quello
dell'**architettura**. Qui tutti i modelli ricevono lo **stesso** input 640×640.

## Due correzioni metodologiche
1. **Letterbox, non stretch.** L'immagine 640×480 viene portata a 640×640 con *letterbox*
   (scala isotropa + padding grigio 114), **non** con resize diretto: lo stretch
   distorcerebbe l'aspect-ratio introducendo un *bias geometrico* sulle bounding box.
   I parametri (scala, padding) sono salvati per rimappare le predizioni all'immagine
   originale 640×480, dove vivono le ground truth.
2. **Risoluzione comune.** Lo stesso tensore 640×640 letterboxato è l'**unico** input
   passato a tutti e quattro i wrapper; nessun modello applica un resize interno aggiuntivo.

## Due tabelle
- **Tabella 1 (principale):** metriche a **risoluzione comune 640×640** per tutti.
- **Tabella 2 (riferimento):** metriche a **risoluzione nativa** di ciascun modello
  (YOLO 640, EfficientDet 640, NanoDet 416, PicoDet 320), per scorporare nel report
  finale quanto del divario è dovuto alla risoluzione e quanto all'architettura.

> NanoDet-Plus-m e PicoDet-S hanno ONNX a input **fisso**: per la Tabella 1 servono export
> ONNX a **640** (path configurabili in Cella *Configurazione*). La Tabella 2 usa gli ONNX
> nativi 416 / 320.

Protocollo identico al report precedente: IoU match = 0.5, conf = 0.25, NMS IoU = 0.45,
greedy matching predizione↔GT. Niente latenza/FPS: solo accuratezza.

In [ ]:
# ============================================================
# Cella 1 - Import librerie
# ============================================================
%pip install ultralytics onnxruntime torchmetrics seaborn --quiet

import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import torch
from torchmetrics.detection import MeanAveragePrecision
import onnxruntime as ort
import tensorflow as tf
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cella 1b - Import modulo condiviso boss_eval_utils
# ============================================================
try:
    import boss_eval_utils as beu  # Kaggle: utility script auto-mounted in sys.path
except ModuleNotFoundError:
    import sys
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/lorenzoverdura/boss_eval_utils")
    import boss_eval_utils as beu
print(f"boss_eval_utils caricato da: {beu.__file__}")

In [ ]:
# ============================================================
# Cella 2 - Configurazione (PATH modificabili in cima, niente hardcode altrove)
# ============================================================
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

# ---- PATH MODELLI -------------------------------------------------
# YOLO11n (.pt) ed EfficientDet Lite4 (.tflite) sono nativi a 640: stesso file
# per Tabella 1 (comune) e Tabella 2 (nativa).
YOLO11N_PT          = Path("/kaggle/input/models/ultralytics/yolo11/pytorch/default/1/yolo11n.pt")
EFFICIENTDET_TFLITE = Path("/kaggle/input/models/tensorflow/efficientdet/tflite/lite4-detection-default/2/2.tflite")

# NanoDet-Plus-m / PicoDet-S: ONNX a input FISSO -> due export per modello.
#   *_ONNX_640    -> risoluzione COMUNE  (Tabella 1)  [DA FORNIRE: export a 640]
#   *_ONNX_NATIVE -> risoluzione NATIVA  (Tabella 2)
NANODET_ONNX_640    = Path("/kaggle/input/PLACEHOLDER-nanodet-640/nanodet-plus-m_640.onnx")
NANODET_ONNX_NATIVE = Path("/kaggle/input/models/lorenzoverdura/nanodet-plus-onnx/onnx/default/1/nanodet-plus-m_416.onnx")
PICODET_ONNX_640    = Path("/kaggle/input/PLACEHOLDER-picodet-640/picodet_s_640_coco.onnx")
PICODET_ONNX_NATIVE = Path("/kaggle/input/models/lorenzoverdura/picodet-s-320/onnx/default/1/picodet_s_320_coco.onnx")

# ---- GROUND TRUTH (export Roboflow COCO, stesso split dei notebook precedenti) ----
GT_COCO_DIR = Path("/kaggle/input/datasets/lorenzoverdura/boss-recordings1/BOSS_recordings.coco")

# ---- ACQUISIZIONE FRAME ------------------------------------------
USE_LIVE_CAMERA = False                       # True -> Intel RealSense D435i (pyrealsense2)
TEST_FRAMES_DIR = Path("./test_frames_640x480")  # usata se USE_LIVE_CAMERA=False
ACQ_W, ACQ_H    = 640, 480                    # risoluzione nominale D435i

# ---- COSTANTI ----------------------------------------------------
COMMON_SIZE    = 640                          # risoluzione comune post-letterbox
CONF_THRESHOLD = beu.CONF_THRESHOLD           # 0.25
IOU_THRESHOLD  = beu.IOU_THRESHOLD            # 0.45 (NMS)
MATCH_IOU      = beu.MATCH_IOU                # 0.50 (TP/FP/FN e matching CM)
DEVICE_YOLO    = "0" if torch.cuda.is_available() else "cpu"

# ---- SPAZIO CLASSI: COCO 80 sequenziale (class_id modello = seq_id 0..79) ----
BOSS_CLASSES            = beu.BOSS_CLASSES
NUM_CLASSES             = beu.NUM_CLASSES
MODEL_CLASSES, MODEL_NC = beu.build_coco_model_classes()
MODEL_TO_BOSS           = beu.build_model_to_boss(MODEL_CLASSES)   # {seq_id: boss_id}
COCO_ID_TO_SEQ          = beu.COCO_ID_TO_SEQ                       # coco 1-based -> seq 0-based

# Le 10 classi BOSS effettivamente rilevabili dai modelli COCO (stesso set del report).
mapped_names = sorted({BOSS_CLASSES[b] for b in MODEL_TO_BOSS.values()})

# ---- OUTPUT ------------------------------------------------------
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./TEST_OUTPUT_LOCAL")
DIR_OUT    = OUTPUT_DIR / "TEST_OUTPUT" / "confronto_unificato"
DIR_OUT.mkdir(parents=True, exist_ok=True)

def _boss_label(seq):
    bid = MODEL_TO_BOSS.get(int(seq))
    return BOSS_CLASSES[bid] if bid is not None else MODEL_CLASSES.get(int(seq), str(seq))

print(f"Ambiente:        {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"USE_LIVE_CAMERA: {USE_LIVE_CAMERA}")
print(f"Risoluzione comune: {COMMON_SIZE}x{COMMON_SIZE} (letterbox, no stretch)")
print(f"Classi modello (COCO seq): {MODEL_NC}")
print(f"Classi BOSS rilevabili ({len(mapped_names)}): {mapped_names}")
print(f"Soglie  conf/NMS/match: {CONF_THRESHOLD} / {IOU_THRESHOLD} / {MATCH_IOU}")
print(f"Output: {DIR_OUT}")
for label, p in [("YOLO11n", YOLO11N_PT), ("EfficientDet", EFFICIENTDET_TFLITE),
                 ("NanoDet 640", NANODET_ONNX_640), ("NanoDet nativo", NANODET_ONNX_NATIVE),
                 ("PicoDet 640", PICODET_ONNX_640), ("PicoDet nativo", PICODET_ONNX_NATIVE),
                 ("GT COCO", GT_COCO_DIR)]:
    print(f"  {label:16s} esiste={p.exists()}  {p}")

In [ ]:
# ============================================================
# Cella 3 - Acquisizione frame (live D435i | cartella test | fallback GT)
# ============================================================
# Robusta: se pyrealsense2 non e' installato o la camera non e' collegata NON
# fallisce, logga un warning e ricade sulle immagini di test. Se anche la
# cartella di test e' assente/vuota, usa la prima immagine della GT.

def acquire_frame():
    """Ritorna UN frame BGR ~640x480 (np.ndarray) per il demo della pipeline."""
    if USE_LIVE_CAMERA:
        try:
            import pyrealsense2 as rs
            pipe = rs.pipeline()
            cfg  = rs.config()
            cfg.enable_stream(rs.stream.color, ACQ_W, ACQ_H, rs.format.bgr8, 30)
            pipe.start(cfg)
            try:
                frames = pipe.wait_for_frames()
                color  = np.asanyarray(frames.get_color_frame().get_data())
            finally:
                pipe.stop()
            print("Frame acquisito da Intel RealSense D435i.")
            return color
        except Exception as e:
            print(f"[WARN] RealSense non disponibile ({e}). Uso le immagini di test.")

    if TEST_FRAMES_DIR.exists():
        imgs = sorted(p for p in TEST_FRAMES_DIR.iterdir()
                      if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
        if imgs:
            print(f"Frame letto da {TEST_FRAMES_DIR}: {imgs[0].name}")
            return cv2.imread(str(imgs[0]))
        print(f"[WARN] {TEST_FRAMES_DIR} vuota.")
    else:
        print(f"[WARN] {TEST_FRAMES_DIR} assente.")

    gt_imgs = sorted((GT_COCO_DIR / "test").glob("*.jpg")) if (GT_COCO_DIR / "test").exists() else []
    if gt_imgs:
        print(f"[WARN] Fallback su immagine GT: {gt_imgs[0].name}")
        return cv2.imread(str(gt_imgs[0]))
    raise FileNotFoundError("Nessuna sorgente frame disponibile (camera / test_frames / GT).")

In [ ]:
# ============================================================
# Cella 4 - Interfaccia unificata + helper di decode GFL
# ============================================================
# Tutti i wrapper espongono la STESSA interfaccia:
#   predict(image)        : image = frame BGR letterboxato 640x640 (input COMUNE).
#                           Ritorna list[dict]
#                             {'box':[x1,y1,x2,y2], 'score':float, 'class_id':int}
#                           in coordinate pixel dell'immagine 640x640 (post-letterbox);
#                           class_id = indice COCO sequenziale 0..79.
#   predict_native(img)   : img = frame BGR originale. Ritorna (boxes_xyxy, seq_ids, scores)
#                           in coordinate dell'immagine ORIGINALE, alla risoluzione NATIVA
#                           del modello (solo Tabella 2).

class DetectorWrapper:
    name = "base"
    native_size = None
    has_common = False
    has_native = False
    def predict(self, image):
        raise NotImplementedError
    def predict_native(self, img_bgr):
        raise NotImplementedError


# --- Decode Generalized Focal Loss (NanoDet-Plus / PicoDet), parametrico ---
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def _softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _make_centers(input_h, input_w, strides, cell_offset):
    """Centri (cx,cy) e stride per punto griglia; ordine livelli = strides,
    intra-livello x veloce / y lento (meshgrid 'ij' row-major)."""
    centers, strs = [], []
    for s in strides:
        fh = int(math.ceil(input_h / s)); fw = int(math.ceil(input_w / s))
        ys, xs = np.meshgrid(np.arange(fh), np.arange(fw), indexing="ij")
        cx = (xs.reshape(-1) + cell_offset) * s
        cy = (ys.reshape(-1) + cell_offset) * s
        centers.append(np.stack([cx, cy], axis=1))
        strs.append(np.full(fh * fw, s, dtype=np.float64))
    return np.concatenate(centers, 0).astype(np.float64), np.concatenate(strs, 0)

def _assemble_gfl(outs, num_classes, reg_bins):
    """Output ONNX -> (cls_scores [N,C], reg [N,reg_bins]); gestisce output
    singolo concatenato, due output (cls/reg), o per-livello."""
    cls_parts, reg_parts, single = [], [], None
    for o in outs:
        a = np.asarray(o)
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        if a.ndim != 2:
            continue
        if a.shape[1] == num_classes:
            cls_parts.append(a)
        elif a.shape[1] == reg_bins:
            reg_parts.append(a)
        elif a.shape[1] == num_classes + reg_bins:
            single = a
    if single is not None:
        return single[:, :num_classes], single[:, num_classes:]
    if cls_parts and reg_parts:
        return np.concatenate(cls_parts, 0), np.concatenate(reg_parts, 0)
    raise RuntimeError("Output ONNX GFL non riconosciuto. Shapes="
                       + str([np.asarray(o).shape for o in outs]))

def _decode_gfl(cls_scores, reg, centers, strs, reg_max, cls_sigmoid):
    """GFL -> (boxes_xyxy_net [N,4], scores [N,C]) in coordinate rete."""
    if cls_sigmoid:
        cls_scores = _sigmoid(cls_scores)
    n = reg.shape[0]
    reg = _softmax(reg.reshape(n, 4, reg_max + 1), axis=-1)
    proj = np.arange(reg_max + 1, dtype=np.float64)
    dist = (reg * proj).sum(-1) * strs[:, None]          # [N,4] pixel l,t,r,b
    x1 = centers[:, 0] - dist[:, 0]; y1 = centers[:, 1] - dist[:, 1]
    x2 = centers[:, 0] + dist[:, 2]; y2 = centers[:, 1] + dist[:, 3]
    return np.stack([x1, y1, x2, y2], axis=1), cls_scores

def _nms_per_class(boxes, labels, scores, iou_thr):
    fb, fl, fs = [], [], []
    for c in np.unique(labels):
        m = labels == c
        bb, ss = boxes[m], scores[m]
        if len(bb) == 0:
            continue
        rects = [[float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])] for b in bb]
        idxs = cv2.dnn.NMSBoxes(rects, [float(s) for s in ss], 0.0, float(iou_thr))
        if idxs is None or len(idxs) == 0:
            continue
        for i in np.array(idxs).reshape(-1):
            fb.append(bb[int(i)]); fl.append(int(c)); fs.append(float(ss[int(i)]))
    if fb:
        return (np.array(fb, dtype=np.float64),
                np.array(fl, dtype=np.int64),
                np.array(fs, dtype=np.float64))
    return (np.zeros((0, 4)), np.array([], dtype=np.int64), np.array([]))

In [ ]:
# ============================================================
# Cella 5 - YOLO11nDetector (ultralytics)
# ============================================================
# YOLO11n nativo = 640. class_id ultralytics (COCO-80 contiguo) == seq_id.
# Su input gia 640x640 il letterbox interno di Ultralytics e' l'identita
# (ratio 1, pad 0) -> nessun resize aggiuntivo.

class YOLO11nDetector(DetectorWrapper):
    name = "YOLO11n"
    native_size = 640
    def __init__(self, pt_path, imgsz=640, nms_iou=IOU_THRESHOLD):
        self.imgsz = imgsz
        self.nms_iou = nms_iou
        self.model = YOLO(str(pt_path))
        self.has_common = True
        self.has_native = True

    def _run(self, source, imgsz):
        res = self.model.predict(source=source, conf=0.0, iou=self.nms_iou,
                                 imgsz=imgsz, device=DEVICE_YOLO, verbose=False)[0]
        if res.boxes is None or len(res.boxes) == 0:
            return (np.zeros((0, 4), np.float32), np.array([], np.int64), np.array([], np.float32))
        return (res.boxes.xyxy.cpu().numpy(),
                res.boxes.cls.cpu().numpy().astype(np.int64),
                res.boxes.conf.cpu().numpy())

    def predict(self, image):
        boxes, ids, sc = self._run(image, 640)          # box gia in spazio 640x640
        return [{"box": [float(b[0]), float(b[1]), float(b[2]), float(b[3])],
                 "score": float(s), "class_id": int(c)}
                for b, c, s in zip(boxes, ids, sc) if int(c) >= 0]

    def predict_native(self, img_bgr):
        # Ultralytics letterboxa internamente e rimappa le box all'originale.
        return self._run(img_bgr, self.imgsz)

In [ ]:
# ============================================================
# Cella 6 - EfficientDetLite4Detector (TFLite)
# ============================================================
# EfficientDet Lite4 input = 640x640. Output identificato per shape (robusto ai
# nomi). class_id 0-based -> +1 -> COCO 1-based -> seq_id. Su input gia 640 non
# c'e' resize aggiuntivo. Per la Tabella 1 si passa il letterbox COMUNE (non lo
# stretch del notebook originale): coerente con gli altri modelli.

class EfficientDetLite4Detector(DetectorWrapper):
    name = "EfficientDet Lite4"
    def __init__(self, tflite_path):
        self.interp = tf.lite.Interpreter(model_path=str(tflite_path))
        self.interp.allocate_tensors()
        self.inp = self.interp.get_input_details()
        out = self.interp.get_output_details()
        self.in_h = int(self.inp[0]["shape"][1]); self.in_w = int(self.inp[0]["shape"][2])
        self.in_dtype = self.inp[0]["dtype"]
        self.idx_boxes = self.idx_classes = self.idx_scores = self.idx_num = None
        two = []
        for d in sorted(out, key=lambda x: x["index"]):
            s = tuple(d["shape"])
            if len(s) == 3 and s[-1] == 4:
                self.idx_boxes = d["index"]
            elif s == (1,) or s == (1, 1):
                self.idx_num = d["index"]
            elif len(s) == 2 and s[0] == 1 and s[1] > 1:
                two.append(d["index"])
        if len(two) >= 2:
            self.idx_classes, self.idx_scores = two[0], two[1]
        elif len(two) == 1:
            self.idx_classes = self.idx_scores = two[0]
        assert None not in (self.idx_boxes, self.idx_classes, self.idx_scores, self.idx_num), \
            "EfficientDet: impossibile identificare i tensori di output"
        self.native_size = self.in_h
        self.has_common = (self.in_h == 640 and self.in_w == 640)
        self.has_native = True

    def _infer(self, canvas_bgr):
        """canvas_bgr: immagine BGR quadrata; ritorna (boxes_xyxy in spazio in_w x in_h,
        seq_ids, scores). Nessun resize se canvas e' gia in_h x in_w."""
        rgb = cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB)
        if rgb.shape[0] != self.in_h or rgb.shape[1] != self.in_w:
            rgb = cv2.resize(rgb, (self.in_w, self.in_h))
        inp = rgb.astype(self.in_dtype)
        if self.in_dtype == np.float32:
            inp = inp / 255.0
        self.interp.set_tensor(self.inp[0]["index"], inp[np.newaxis])
        self.interp.invoke()
        n = int(self.interp.get_tensor(self.idx_num).flat[0])
        boxes   = self.interp.get_tensor(self.idx_boxes)[0][:n]     # ymin xmin ymax xmax norm
        classes = self.interp.get_tensor(self.idx_classes)[0][:n]
        scores  = self.interp.get_tensor(self.idx_scores)[0][:n]
        ymin, xmin, ymax, xmax = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        bx = np.stack([xmin*self.in_w, ymin*self.in_h, xmax*self.in_w, ymax*self.in_h], axis=1)
        coco = classes.astype(int) + 1
        seq  = np.array([COCO_ID_TO_SEQ.get(int(c), -1) for c in coco], dtype=np.int64)
        return bx, seq, scores

    def predict(self, image):
        bx, seq, sc = self._infer(image)                 # box in spazio 640x640
        return [{"box": [float(b[0]), float(b[1]), float(b[2]), float(b[3])],
                 "score": float(s), "class_id": int(c)}
                for b, c, s in zip(bx, seq, sc) if int(c) >= 0]

    def predict_native(self, img_bgr):
        canvas, ratio, pad = beu.letterbox(img_bgr, (self.in_h, self.in_w))
        bx, seq, sc = self._infer(canvas)
        bx = beu.scale_boxes_from_letterbox(bx, ratio, pad, img_bgr.shape[:2])
        valid = seq >= 0
        return bx[valid], seq[valid], sc[valid]

In [ ]:
# ============================================================
# Cella 7 - GFLDetector: NanoDet-Plus-m e PicoDet-S (ONNX, testa GFL)
# ============================================================
# Stesso schema di testa (Generalized Focal Loss). Due sessioni ONNX per modello:
#   - comune (640)  usata da predict()         -> Tabella 1
#   - nativa (416/320) usata da predict_native -> Tabella 2
# I parametri di decode (strides, reg_max, offset, normalizzazione, cls_sigmoid)
# sono identici tra le due risoluzioni: dipendono dal modello, non dall'input.

class GFLDetector(DetectorWrapper):
    def __init__(self, name, common_onnx, native_onnx, native_size,
                 strides, reg_max, cell_offset, to_rgb, norm_scale,
                 norm_mean, norm_std, cls_sigmoid,
                 score_floor=0.05, nms_iou=IOU_THRESHOLD, num_classes=80):
        self.name = name
        self.native_size = native_size
        self.strides = strides
        self.reg_max = reg_max
        self.reg_bins = 4 * (reg_max + 1)
        self.cell_offset = cell_offset
        self.to_rgb = to_rgb
        self.norm_scale = norm_scale
        self.norm_mean = norm_mean
        self.norm_std = norm_std
        self.cls_sigmoid = cls_sigmoid
        self.score_floor = score_floor
        self.nms_iou = nms_iou
        self.num_classes = num_classes

        self.sess_c = self.in_c = self.centers_c = self.strs_c = None
        if common_onnx is not None and Path(common_onnx).exists():
            self.sess_c, self.in_c, self.centers_c, self.strs_c = self._load(common_onnx)
            self.has_common = (self.in_c[0] == 640 and self.in_c[1] == 640)

        self.sess_n = self.in_n = self.centers_n = self.strs_n = None
        if native_onnx is not None and Path(native_onnx).exists():
            self.sess_n, self.in_n, self.centers_n, self.strs_n = self._load(native_onnx)
            self.has_native = True

    def _load(self, onnx_path):
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ish = sess.get_inputs()[0].shape
        h = int(ish[2]) if isinstance(ish[2], int) else self.native_size
        w = int(ish[3]) if isinstance(ish[3], int) else self.native_size
        centers, strs = _make_centers(h, w, self.strides, self.cell_offset)
        return sess, (h, w, sess.get_inputs()[0].name), centers, strs

    def _preprocess(self, canvas_bgr, in_hw):
        h, w, _ = in_hw
        img = canvas_bgr
        if img.shape[0] != h or img.shape[1] != w:
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32)
        if self.to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img * self.norm_scale
        img = (img - self.norm_mean) / self.norm_std
        return np.ascontiguousarray(img.transpose(2, 0, 1)[None].astype(np.float32))

    def _infer(self, canvas_bgr, sess, in_hw, centers, strs):
        blob = self._preprocess(canvas_bgr, in_hw)
        outs = sess.run(None, {in_hw[2]: blob})
        cls, reg = _assemble_gfl(outs, self.num_classes, self.reg_bins)
        if cls.shape[0] != len(centers):
            raise RuntimeError(f"{self.name}: mismatch punti {cls.shape[0]} vs griglia "
                               f"{len(centers)} - verifica strides/reg_max/input.")
        boxes_net, scores_all = _decode_gfl(cls, reg, centers, strs, self.reg_max, self.cls_sigmoid)
        labels = scores_all.argmax(1).astype(np.int64)
        conf = scores_all.max(1)
        keep = conf >= self.score_floor
        return boxes_net[keep], labels[keep], conf[keep]

    def predict(self, image):
        # image = letterbox COMUNE 640; sessione 640 -> nessun letterbox aggiuntivo.
        b, l, s = self._infer(image, self.sess_c, self.in_c, self.centers_c, self.strs_c)
        b, l, s = _nms_per_class(b, l, s, self.nms_iou)
        return [{"box": [float(x[0]), float(x[1]), float(x[2]), float(x[3])],
                 "score": float(sc), "class_id": int(c)}
                for x, c, sc in zip(b, l, s) if int(c) >= 0]

    def predict_native(self, img_bgr):
        canvas, ratio, pad = beu.letterbox(img_bgr, (self.in_n[0], self.in_n[1]))
        b, l, s = self._infer(canvas, self.sess_n, self.in_n, self.centers_n, self.strs_n)
        b = beu.scale_boxes_from_letterbox(b, ratio, pad, img_bgr.shape[:2])
        return _nms_per_class(b, l, s, self.nms_iou)

In [ ]:
# ============================================================
# Cella 8 - Istanziazione wrapper (skip robusto se un file manca)
# ============================================================
wrappers = []

def _try(ctor, label):
    try:
        w = ctor(); wrappers.append(w)
        print(f"OK   {label:18s} common={w.has_common}  native={w.has_native}")
    except Exception as e:
        print(f"SKIP {label:18s} {e}")

_try(lambda: YOLO11nDetector(YOLO11N_PT), "YOLO11n")
_try(lambda: EfficientDetLite4Detector(EFFICIENTDET_TFLITE), "EfficientDet Lite4")
_try(lambda: GFLDetector("NanoDet-Plus-m", NANODET_ONNX_640, NANODET_ONNX_NATIVE, 416,
                         strides=(8, 16, 32, 64), reg_max=7, cell_offset=0.0, to_rgb=False,
                         norm_scale=1.0,
                         norm_mean=np.array([103.53, 116.28, 123.675], dtype=np.float32),
                         norm_std=np.array([57.375, 57.12, 58.395], dtype=np.float32),
                         cls_sigmoid=False), "NanoDet-Plus-m")
_try(lambda: GFLDetector("PicoDet-S", PICODET_ONNX_640, PICODET_ONNX_NATIVE, 320,
                         strides=(8, 16, 32, 64), reg_max=7, cell_offset=0.5, to_rgb=True,
                         norm_scale=1.0/255.0,
                         norm_mean=np.array([0.485, 0.456, 0.406], dtype=np.float32),
                         norm_std=np.array([0.229, 0.224, 0.225], dtype=np.float32),
                         cls_sigmoid=False), "PicoDet-S")

print(f"\nWrapper attivi: {[w.name for w in wrappers]}")
print(f"Disponibili per Tabella 1 (640): {[w.name for w in wrappers if w.has_common]}")
print(f"Disponibili per Tabella 2 (nativa): {[w.name for w in wrappers if w.has_native]}")

In [ ]:
# ============================================================
# Cella 9 - Demo pipeline unificata su UN frame (letterbox 640 condiviso)
# ============================================================
# Stesso identico tensore 640x640 passato a tutti i wrapper disponibili.
demo_orig = acquire_frame()
print(f"Frame acquisito: {demo_orig.shape[1]}x{demo_orig.shape[0]} (WxH)")

# Letterbox COMUNE: scala isotropa + padding (NO stretch -> niente bias geometrico).
demo_canvas, demo_ratio, demo_pad = beu.letterbox(demo_orig, COMMON_SIZE)

active = [w for w in wrappers if w.has_common]
if active:
    cols = len(active)
    fig, axes = plt.subplots(1, cols, figsize=(6 * cols, 6))
    axes = np.atleast_1d(axes)
    for ax, w in zip(axes, active):
        dets = w.predict(demo_canvas)                # box in spazio 640x640
        vis = demo_canvas.copy()
        for d in dets:
            if d["score"] < CONF_THRESHOLD:
                continue
            x1, y1, x2, y2 = [int(v) for v in d["box"]]
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(vis, f'{_boss_label(d["class_id"])} {d["score"]:.2f}',
                        (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        n_show = sum(1 for d in dets if d["score"] >= CONF_THRESHOLD)
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{w.name}  ({n_show} det @ conf>={CONF_THRESHOLD})", fontsize=11)
        ax.axis("off")
    plt.suptitle("Demo pipeline unificata - stesso letterbox 640x640 per tutti i modelli", fontsize=14)
    plt.tight_layout()
    save_path = DIR_OUT / "demo_pipeline_640.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato: {save_path}")
else:
    print("[WARN] Nessun wrapper disponibile a 640: fornire gli ONNX 640 di NanoDet/PicoDet.")

In [ ]:
# ============================================================
# Cella 10 - Ground Truth (COCO JSON Roboflow) + mappa categorie -> seq_id
# ============================================================
gt_json_path = GT_COCO_DIR / "test" / "_annotations.coco.json"
GT_TEST_IMGS = GT_COCO_DIR / "test"
if not gt_json_path.exists():
    raise FileNotFoundError(f"Annotazioni COCO non trovate: {gt_json_path}")

with open(gt_json_path) as f:
    gt_data = json.load(f)
print(f"GT COCO: {len(gt_data['images'])} immagini, {len(gt_data['annotations'])} annotazioni, "
      f"{len(gt_data['categories'])} categorie")

# category_id Roboflow -> seq_id (0..79) via classe BOSS canonica.
cat_id_to_seq, unmapped_cats = {}, []
for cat in gt_data["categories"]:
    boss = beu.resolve_to_boss(cat["name"])
    seq = None
    if boss is not None:
        for seq_id, coco_name in MODEL_CLASSES.items():
            if beu.resolve_to_boss(coco_name) == boss:
                seq = seq_id
                break
    (cat_id_to_seq.__setitem__(cat["id"], seq) if seq is not None
     else unmapped_cats.append(cat["name"]))
print(f"Categorie GT mappate (cat_id -> seq_id): {cat_id_to_seq}")
print(f"Categorie GT scartate (non rilevabili): {unmapped_cats}")

img_id_to_anns = {}
for ann in gt_data["annotations"]:
    img_id_to_anns.setdefault(ann["image_id"], []).append(ann)

In [ ]:
# ============================================================
# Cella 11 - Confusion matrix + funzione di valutazione
# ============================================================
# Protocollo (identico al report): IoU match = MATCH_IOU, conf = CONF_THRESHOLD,
# NMS IoU = IOU_THRESHOLD, greedy matching predizione<->GT. mAP@0.5 via
# torchmetrics; Precision/Recall reali via TP/FP/FN (beu).

# --- Confusion matrix sulle 10 classi BOSS coperte + 'background' ---
CM_CLASSES = mapped_names                         # 10 nomi BOSS
N_CM = len(CM_CLASSES) + 1
BG = len(CM_CLASSES)                              # indice 'background'
boss_name_to_cm = {n: i for i, n in enumerate(CM_CLASSES)}

def _seq_to_cm(sid):
    bid = MODEL_TO_BOSS.get(int(sid))
    return boss_name_to_cm.get(BOSS_CLASSES[bid]) if bid is not None else None

def update_confusion(cm, pred_boxes, pred_seq, pred_scores, gt_boxes, gt_labels, iou_thr):
    """CM[riga=GT, col=Pred] + riga/col 'background' per FN/FP. Greedy per IoU."""
    p = [(b, _seq_to_cm(s), sc) for b, s, sc in zip(pred_boxes, pred_seq, pred_scores)]
    p = [t for t in p if t[1] is not None]
    g = [(b, _seq_to_cm(s)) for b, s in zip(gt_boxes, gt_labels)]
    g = [t for t in g if t[1] is not None]
    gb = np.array([b for b, _ in g], dtype=np.float64).reshape(-1, 4)
    gi = [i for _, i in g]
    matched = np.zeros(len(g), dtype=bool)
    for k in sorted(range(len(p)), key=lambda k: -p[k][2]):     # pred per score decrescente
        b, pi, _ = p[k]
        if len(g):
            ious = beu.iou_one_to_many(np.array(b, dtype=np.float64), gb).copy()
            ious[matched] = -1.0
            j = int(ious.argmax())
            if ious[j] >= iou_thr:
                matched[j] = True
                cm[gi[j], pi] += 1                 # match (diagonale se classe corretta)
                continue
        cm[BG, pi] += 1                            # falso positivo
    for j in range(len(g)):
        if not matched[j]:
            cm[gi[j], BG] += 1                     # falso negativo (oggetto perso)
    return cm

def evaluate(infer_fn, desc="eval"):
    """infer_fn(img_bgr_orig) -> (boxes_xyxy_orig, seq_ids, scores). Ritorna dict
    con map50, aggregate, per_class df e confusion matrix."""
    metric50 = MeanAveragePrecision(iou_type="bbox", class_metrics=True, iou_thresholds=[0.5])
    tp = np.zeros(MODEL_NC, np.int64); fp = np.zeros(MODEL_NC, np.int64); fn = np.zeros(MODEL_NC, np.int64)
    cm = np.zeros((N_CM, N_CM), np.int64)
    processed = 0
    for img_info in tqdm(gt_data["images"], desc=desc):
        img = cv2.imread(str(GT_TEST_IMGS / img_info["file_name"]))
        if img is None:
            continue
        gt_boxes, gt_labels = [], []
        for ann in img_id_to_anns.get(img_info["id"], []):
            sid = cat_id_to_seq.get(ann["category_id"])
            if sid is None:
                continue
            x, y, bw, bh = ann["bbox"]
            gt_boxes.append([x, y, x + bw, y + bh]); gt_labels.append(sid)
        if not gt_boxes:
            continue

        boxes, seq, scores = infer_fn(img)
        valid = seq >= 0
        preds = [{"boxes":  torch.tensor(boxes[valid], dtype=torch.float32),
                  "scores": torch.tensor(scores[valid], dtype=torch.float32),
                  "labels": torch.tensor(seq[valid],   dtype=torch.int64)}]
        targets = [{"boxes":  torch.tensor(gt_boxes,  dtype=torch.float32),
                    "labels": torch.tensor(gt_labels, dtype=torch.int64)}]
        metric50.update(preds, targets)

        keep = valid & (scores >= CONF_THRESHOLD)
        beu.accumulate_matches(tp, fp, fn, boxes[keep], seq[keep], scores[keep],
                               gt_boxes, gt_labels, MATCH_IOU)
        update_confusion(cm, boxes[keep], seq[keep], scores[keep], gt_boxes, gt_labels, MATCH_IOU)
        processed += 1

    res = metric50.compute()
    map50 = float(res["map"])
    classes_t = res["classes"].numpy().astype(int)
    ap_t = res["map_per_class"].numpy()
    prec, rec = beu.precision_recall_per_class(tp, fp, fn)

    rows = []
    for ci, ap in zip(classes_t, ap_t):
        if ap < 0:
            continue
        p_, r_ = float(prec[ci]), float(rec[ci])
        rows.append({"Classe": _boss_label(ci), "AP@0.5": float(ap),
                     "Precision": p_, "Recall": r_,
                     "F1": 2 * p_ * r_ / (p_ + r_ + 1e-9)})
    per_class = pd.DataFrame(rows)

    P = float(per_class["Precision"].mean()) if len(per_class) else 0.0
    R = float(per_class["Recall"].mean()) if len(per_class) else 0.0
    agg = {"mAP@0.5": map50, "Precision": P, "Recall": R,
           "F1": 2 * P * R / (P + R + 1e-9)}
    return {"map50": map50, "agg": agg, "per_class": per_class,
            "cm": cm, "processed": processed}

def make_common_infer(w):
    def f(orig):
        canvas, ratio, pad = beu.letterbox(orig, COMMON_SIZE)
        dets = w.predict(canvas)
        if dets:
            boxes = np.array([d["box"] for d in dets], dtype=np.float64)
            seq   = np.array([d["class_id"] for d in dets], dtype=np.int64)
            sc    = np.array([d["score"] for d in dets], dtype=np.float64)
        else:
            boxes, seq, sc = np.zeros((0, 4)), np.array([], np.int64), np.array([])
        boxes = beu.scale_boxes_from_letterbox(boxes, ratio, pad, orig.shape[:2])
        return boxes, seq, sc
    return f

def make_native_infer(w):
    return lambda orig: w.predict_native(orig)

In [ ]:
# ============================================================
# Cella 12 - Esecuzione valutazione: ogni modello x {comune 640, nativa}
# ============================================================
results = {}
for w in wrappers:
    if w.has_common:
        results[(w.name, "common")] = evaluate(make_common_infer(w), desc=f"{w.name} @640")
    if w.has_native:
        results[(w.name, "native")] = evaluate(make_native_infer(w), desc=f"{w.name} nativa")

print("\nValutazioni completate:")
for (name, exp), r in results.items():
    a = r["agg"]
    print(f"  {name:18s} [{exp:6s}]  mAP@0.5={a['mAP@0.5']:.4f}  "
          f"P={a['Precision']:.4f}  R={a['Recall']:.4f}  F1={a['F1']:.4f}  "
          f"(img={r['processed']})")

In [ ]:
# ============================================================
# Cella 13 - Tabella 1 (risoluzione comune 640) e Tabella 2 (nativa) + per-classe
# ============================================================
def build_table(exp):
    rows = []
    for w in wrappers:
        r = results.get((w.name, exp))
        if r is None:
            continue
        a = r["agg"]
        size = COMMON_SIZE if exp == "common" else w.native_size
        rows.append({"Modello": w.name, "Input": f"{size}x{size}",
                     "mAP@0.5": a["mAP@0.5"], "Precision": a["Precision"],
                     "Recall": a["Recall"], "F1": a["F1"]})
    df = pd.DataFrame(rows).sort_values("mAP@0.5", ascending=False).reset_index(drop=True)
    return df

df_common = build_table("common")
df_native = build_table("native")

fmt = {c: "{:.4f}".format for c in ["mAP@0.5", "Precision", "Recall", "F1"]}

print("=" * 70)
print("TABELLA 1 - Risoluzione COMUNE 640x640 (post-letterbox) per tutti i modelli")
print("=" * 70)
print(df_common.to_string(index=False, formatters=fmt))
df_common.to_csv(DIR_OUT / "tabella1_risoluzione_comune_640.csv", index=False)

print("\n" + "=" * 70)
print("TABELLA 2 - Risoluzione NATIVA di ciascun modello (riferimento)")
print("=" * 70)
print(df_native.to_string(index=False, formatters=fmt))
df_native.to_csv(DIR_OUT / "tabella2_risoluzione_nativa.csv", index=False)

# --- Metriche per classe (risoluzione comune) ---
print("\n" + "=" * 70)
print("METRICHE PER CLASSE - Risoluzione comune 640x640")
print("=" * 70)
for w in wrappers:
    r = results.get((w.name, "common"))
    if r is None or r["per_class"].empty:
        continue
    print(f"\n--- {w.name} ---")
    dfc = r["per_class"].copy()
    for c in ["AP@0.5", "Precision", "Recall", "F1"]:
        dfc[c] = dfc[c].map("{:.4f}".format)
    print(dfc.to_string(index=False))
    r["per_class"].to_csv(DIR_OUT / f"per_classe_640_{w.name.replace(' ', '_')}.csv", index=False)

print(f"\nCSV salvati in: {DIR_OUT}")

In [ ]:
# ============================================================
# Cella 14 - Confusion matrix per modello (risoluzione comune 640)
# ============================================================
# Deliverable D4.3.2: una confusion matrix per modello. Righe = GT, colonne =
# predizione, con riga/colonna 'background' per falsi negativi / falsi positivi.
labels_cm = CM_CLASSES + ["background"]
common_models = [w for w in wrappers if (w.name, "common") in results]

if common_models:
    n = len(common_models)
    cols = min(2, n)
    rowsfig = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rowsfig, cols, figsize=(9 * cols, 7.5 * rowsfig))
    axes = np.atleast_1d(axes).ravel()
    for ax, w in zip(axes, common_models):
        cm = results[(w.name, "common")]["cm"]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=labels_cm, yticklabels=labels_cm, ax=ax)
        ax.set_title(f"Confusion Matrix - {w.name} (640x640)", fontsize=12)
        ax.set_xlabel("Predetto"); ax.set_ylabel("Ground Truth")
        ax.tick_params(axis="x", rotation=45); ax.tick_params(axis="y", rotation=0)
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    save_path = DIR_OUT / "confusion_matrix_640.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato: {save_path}")

    # Salvataggio singolo CSV per modello.
    for w in common_models:
        cm = results[(w.name, "common")]["cm"]
        pd.DataFrame(cm, index=labels_cm, columns=labels_cm).to_csv(
            DIR_OUT / f"confusion_matrix_640_{w.name.replace(' ', '_')}.csv")
else:
    print("[WARN] Nessun risultato a risoluzione comune: confusion matrix non generate.")

In [ ]:
# ============================================================
# Cella 15 - Note finali: cambia il ranking nativa -> comune?
# ============================================================
# Domanda chiave: a parita' di risoluzione (640), il verdetto del report
# precedente (YOLO11n come modello scelto) regge o cambia?

def ranking(exp):
    items = [(w.name, results[(w.name, exp)]["agg"]["mAP@0.5"])
             for w in wrappers if (w.name, exp) in results]
    return sorted(items, key=lambda x: -x[1])

rank_native = ranking("native")
rank_common = ranking("common")

lines = ["# Note finali - risoluzione nativa vs comune (mAP@0.5)\n"]
lines.append("## Ranking a risoluzione NATIVA")
for i, (n, m) in enumerate(rank_native, 1):
    size = next(w.native_size for w in wrappers if w.name == n)
    lines.append(f"{i}. {n} ({size}x{size}) - mAP@0.5 = {m:.4f}")
lines.append("\n## Ranking a risoluzione COMUNE 640x640")
for i, (n, m) in enumerate(rank_common, 1):
    lines.append(f"{i}. {n} - mAP@0.5 = {m:.4f}")

# Variazione di posizione per modello presente in entrambi.
pos_n = {n: i for i, (n, _) in enumerate(rank_native, 1)}
pos_c = {n: i for i, (n, _) in enumerate(rank_common, 1)}
lines.append("\n## Variazione di posizione (nativa -> comune)")
for n in pos_c:
    if n in pos_n:
        d = pos_n[n] - pos_c[n]
        arrow = "invariata" if d == 0 else (f"sale di {d}" if d > 0 else f"scende di {-d}")
        lines.append(f"- {n}: pos {pos_n[n]} -> {pos_c[n]} ({arrow})")
    else:
        lines.append(f"- {n}: assente a risoluzione nativa")

# Verdetto sintetico.
lines.append("\n## Verdetto")
if rank_common and rank_native:
    top_c, top_n = rank_common[0][0], rank_native[0][0]
    if top_c == top_n:
        lines.append(f"Il modello migliore resta **{top_c}** anche a parita' di risoluzione: "
                     f"il verdetto del report precedente e' CONFERMATO.")
    else:
        lines.append(f"A risoluzione comune il migliore diventa **{top_c}** (era **{top_n}**): "
                     f"il verdetto del report precedente va RIVALUTATO - parte del divario "
                     f"era dovuto alla risoluzione, non solo all'architettura.")
    if "YOLO11n" in pos_c:
        lines.append(f"YOLO11n: posizione a risoluzione comune = {pos_c['YOLO11n']} su {len(pos_c)}.")
else:
    lines.append("Dati insufficienti: fornire gli ONNX 640 di NanoDet/PicoDet per completare la Tabella 1.")

report_txt = "\n".join(lines)
print(report_txt)
(DIR_OUT / "note_finali_ranking.md").write_text(report_txt, encoding="utf-8")
print(f"\nSalvato: {DIR_OUT / 'note_finali_ranking.md'}")